<!-- NB_DOC_INTRO_V1 -->
# NLP — Classification de texte sur classes desequilibrees

> 📚 **Doc thematique** : [docs/05_NLP.md](docs/05_NLP.md) (NLP)
> 📖 **Inventaire** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md) | 🗂️ **README** : [README.md](README.md)

---

## Presentation

**Cas reel** : ton dataset texte a 95% de la classe 0 et 5% de la classe 1. L'accuracy par default trompe : un modele 'toujours 0' atteint 95% ! Ce notebook execute les **techniques anti-desequilibre** sur du texte : `class_weight='balanced'`, undersampling, oversampling, **SMOTE** (sur vecteurs TF-IDF), back-translation (augmentation NLP), threshold tuning.

## Plan

1. Setup + dataset desequilibre simule
2. Baseline naive (LogReg sans correction)
3. class_weight='balanced'
4. Undersampling (RandomUnderSampler)
5. Oversampling (RandomOverSampler)
6. SMOTE sur TF-IDF (discutable mais teste)
7. Threshold tuning (precision/recall)
8. Bench complet
9. Augmentation NLP (concepts : back-translation, EDA, nlpaug)
10. Pieges + References


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score, precision_recall_curve,
    average_precision_score,
)
import warnings
warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

## 1. Dataset desequilibre (95/5)

In [ ]:
# 20 Newsgroups subset binaire desequilibre
data = fetch_20newsgroups(subset="all", categories=["sci.med", "sci.space"],
                           remove=("headers", "footers", "quotes"), random_state=SEED)

# Garder TOUS les sci.med (~1000) + 5% seulement de sci.space (rare)
idx_med = np.where(np.array(data.target) == data.target_names.index("sci.med"))[0]
idx_space = np.where(np.array(data.target) == data.target_names.index("sci.space"))[0]
np.random.seed(SEED)
idx_space_sub = np.random.choice(idx_space, size=len(idx_med) // 20, replace=False)
indices = np.concatenate([idx_med, idx_space_sub])

texts = [data.data[i] for i in indices]
labels = np.array([0 if data.target[i] == data.target_names.index("sci.med") else 1 for i in indices])
print(f"N total : {len(texts)}")
print(f"Distribution : class 0 = {(labels == 0).sum()}, class 1 = {(labels == 1).sum()}")
print(f"Ratio : {labels.mean():.3f} (classe 1 = MINORITAIRE = 'space')")

In [ ]:
# Train/test stratifie
from sklearn.model_selection import train_test_split
texts_tr, texts_te, y_tr, y_te = train_test_split(texts, labels, test_size=0.25, stratify=labels, random_state=SEED)

# Vectoriser
tfidf = TfidfVectorizer(stop_words="english", max_features=5_000, min_df=2)
X_tr = tfidf.fit_transform(texts_tr)
X_te = tfidf.transform(texts_te)
print(f"X_tr shape : {X_tr.shape}, y_tr balance : {y_tr.mean():.3f}")

## 2. Baseline naive

In [ ]:
clf = LogisticRegression(max_iter=1000, random_state=SEED).fit(X_tr, y_tr)
pred = clf.predict(X_te)
print("=== Baseline (sans correction) ===")
print(classification_report(y_te, pred, target_names=["med (maj)", "space (min)"], digits=3))
print(f"F1 macro : {f1_score(y_te, pred, average='macro'):.4f}")
print(f"\nConfusion :\n{confusion_matrix(y_te, pred)}")

Note : on voit que la classe minoritaire (space) a un recall faible — le modele privilegie la majoritaire.

## 3. `class_weight='balanced'` — la solution la plus simple

In [ ]:
clf_w = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED).fit(X_tr, y_tr)
pred_w = clf_w.predict(X_te)
print("=== class_weight='balanced' ===")
print(classification_report(y_te, pred_w, target_names=["med", "space"], digits=3))

## 4. Undersampling et Oversampling avec `imblearn`

In [ ]:
try:
    from imblearn.under_sampling import RandomUnderSampler
    from imblearn.over_sampling import RandomOverSampler, SMOTE

    # === RandomUnderSampler ===
    rus = RandomUnderSampler(random_state=SEED)
    X_us, y_us = rus.fit_resample(X_tr, y_tr)
    print(f"Apres undersampling : {X_us.shape}, balance {y_us.mean():.3f}")
    clf_us = LogisticRegression(max_iter=1000, random_state=SEED).fit(X_us, y_us)
    print(f"  F1 macro test : {f1_score(y_te, clf_us.predict(X_te), average='macro'):.4f}")

    # === RandomOverSampler ===
    ros = RandomOverSampler(random_state=SEED)
    X_os, y_os = ros.fit_resample(X_tr, y_tr)
    print(f"\nApres oversampling : {X_os.shape}, balance {y_os.mean():.3f}")
    clf_os = LogisticRegression(max_iter=1000, random_state=SEED).fit(X_os, y_os)
    print(f"  F1 macro test : {f1_score(y_te, clf_os.predict(X_te), average='macro'):.4f}")

    SMOTE_OK = True
except ImportError:
    print("imblearn pas installe : uv add --group ml imbalanced-learn")
    SMOTE_OK = False

## 5. SMOTE sur TF-IDF

**Discutable** : interpoler des vecteurs TF-IDF cree des 'documents' qui n'existent pas. Mais en pratique, sur des problemes severement desequilibres, ca marche souvent.

In [ ]:
if SMOTE_OK:
    smote = SMOTE(random_state=SEED, k_neighbors=3)
    X_sm, y_sm = smote.fit_resample(X_tr, y_tr)
    print(f"Apres SMOTE : {X_sm.shape}, balance {y_sm.mean():.3f}")
    clf_sm = LogisticRegression(max_iter=1000, random_state=SEED).fit(X_sm, y_sm)
    print(f"SMOTE F1 macro test : {f1_score(y_te, clf_sm.predict(X_te), average='macro'):.4f}")
else:
    print("SKIP SMOTE")

## 6. Threshold tuning — PR curve + choisir le seuil optimal

In [ ]:
# Probas (au lieu de predict qui utilise threshold 0.5)
proba = clf_w.predict_proba(X_te)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_te, proba)
ap = average_precision_score(y_te, proba)

# Trouver le threshold qui maximise F1
from sklearn.metrics import f1_score
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)
best_idx = np.argmax(f1_scores)
best_thresh = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
print(f"AP (avg precision) : {ap:.4f}")
print(f"Best F1 : {f1_scores[best_idx]:.4f} a threshold = {best_thresh:.3f}")

# Plot PR curve
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(recall, precision, lw=2, label=f"AP={ap:.3f}")
ax.scatter(recall[best_idx], precision[best_idx], color="red", s=80, zorder=5,
           label=f"Best F1 ({best_thresh:.3f})")
ax.axhline(y_te.mean(), color="grey", ls="--", alpha=0.5, label=f"Baseline ({y_te.mean():.2f})")
ax.set(xlabel="Recall", ylabel="Precision", title="PR Curve (class minoritaire)")
ax.legend()
plt.show()

# Predict avec le bon threshold
pred_opt = (proba >= best_thresh).astype(int)
print(f"\n=== Avec threshold {best_thresh:.3f} ===")
print(classification_report(y_te, pred_opt, target_names=["med", "space"], digits=3))

## 7. Augmentation NLP (concepts)

Plutot que SMOTE sur vecteurs, on peut **augmenter le texte** directement :

| Technique | Description | Lib |
|---|---|---|
| **Back-translation** | Texte FR → EN → FR (legere variation) | `googletrans`, `deep-translator` |
| **EDA** (Easy Data Augmentation) | Synonym swap, random insertion/deletion | `nlpaug` |
| **Contextual augmentation** | BERT remplace 1 mot par un plausible | `nlpaug` |
| **TF-IDF replacement** | Remplace mots peu importants | `nlpaug` |
| **Paraphrase generation** | T5 / GPT genere des paraphrases | `transformers` |

```python
# Exemple nlpaug
import nlpaug.augmenter.word as naw
aug = naw.SynonymAug(aug_p=0.1)
augmented = aug.augment(original_text)
```

Plus naturel que SMOTE pour le texte, mais plus lent et necessite de **labelliser** les augmentations (on suppose meme label que l'original).

## 8. Pieges et anti-patterns

| ❌ Anti-pattern | ✅ Correctif |
|---|---|
| Accuracy sur desequilibre 95/5 | F1 macro / AUC / PR-AUC |
| SMOTE sur **tout** (incl. test) | Toujours sur TRAIN uniquement (data leakage sinon) |
| Sub-sampling de la majo → perd info | Cross-validation strategy : combiner methodes |
| Threshold = 0.5 par defaut | Tuner sur PR curve |
| Pas regarder confusion matrix | Identifier FN vs FP |
| Augmentation NLP sans verifier label | Texte modifie → label parfois change ! |
| Choisir methode sans benchmark | Tester 3-4 approches (class_weight, ROS, SMOTE) |


## References

### Documentation
- imbalanced-learn : https://imbalanced-learn.org/
- nlpaug : https://github.com/makcedward/nlpaug
- SMOTE paper : https://www.jair.org/index.php/jair/article/view/10302

### Voir aussi
- [NLP_Classification_Spervisé.ipynb](NLP_Classification_Spervisé.ipynb)
- [DS_Metrics_Reference.ipynb](DS_Metrics_Reference.ipynb)
- [Structures_Preprocessing.ipynb](Structures_Preprocessing.ipynb)
